In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import json
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import load_model

warnings.filterwarnings('ignore')
sns.set()
sns.set_context('talk')

In [ ]:
# ── Data ────────────────────────────────────────────────────────────────────
with open('../../config.json') as f:
    cfg = json.load(f)

target    = cfg['rc']
file_name = cfg['rc_file_name']

data = pd.read_csv(f'../../data/{file_name}.csv')
X = data.drop(columns=[target])
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = joblib.load('saved_models/scaler_rc.pkl')
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns)
X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns)

print(f'Test set: {X_test.shape[0]} samples')

In [ ]:
# ── Load models ──────────────────────────────────────────────────────────────
rf  = joblib.load('saved_models/random_forest_rc.pkl')
xgb = joblib.load('saved_models/xgboost_rc.pkl')
lgb = joblib.load('saved_models/xgboost.pkl')
ann = load_model('saved_models/ann_lr0.002_batch16.keras')

# ── Predictions ──────────────────────────────────────────────────────────────
preds = {
    'Random Forest': rf.predict(X_test),
    'XGBoost':       xgb.predict(X_test),
    'LightGBM':      lgb.predict(X_test),
    'ANN':           ann.predict(X_test_scaled).flatten(),
}

# ── Metrics ──────────────────────────────────────────────────────────────────
rows = []
for name, y_pred in preds.items():
    rows.append({
        'Model': name,
        'R²':    round(r2_score(y_test, y_pred), 4),
        'MAE':   round(mean_absolute_error(y_test, y_pred), 4),
        'RMSE':  round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
    })

metrics_df = pd.DataFrame(rows).set_index('Model')
print(metrics_df)

In [ ]:
# ── 1. Metrics bar chart ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

for ax, metric in zip(axes, ['R²', 'MAE', 'RMSE']):
    bars = ax.bar(metrics_df.index, metrics_df[metric], color=colors)
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.set_xticklabels(metrics_df.index, rotation=15, ha='right')
    for bar, val in zip(bars, metrics_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005 * bar.get_height(),
                f'{val:.3f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Model Comparison — CS (MPa)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('saved_models/comparison_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2. Actual vs Predicted (4 subplots) ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for ax, (name, y_pred), color in zip(axes, preds.items(), colors):
    ax.scatter(y_test, y_pred, alpha=0.6, color=color, edgecolors='white', linewidths=0.5)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
    ax.set_xlabel('Actual CS (MPa)')
    ax.set_ylabel('Predicted CS (MPa)')
    r2  = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    ax.set_title(f'{name}\nR² = {r2:.3f}  |  RMSE = {rmse:.3f}')
    ax.legend(fontsize=9)

plt.suptitle('Actual vs Predicted — CS (MPa)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('saved_models/comparison_actual_vs_pred.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3. Residuals distribution ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (name, y_pred), color in zip(axes, preds.items(), colors):
    residuals = np.array(y_test) - y_pred
    ax.hist(residuals, bins=20, color=color, edgecolor='white', alpha=0.8)
    ax.axvline(0, color='red', linestyle='--', linewidth=1.5)
    ax.set_xlabel('Residual (Actual − Predicted)')
    ax.set_ylabel('Count')
    ax.set_title(f'{name} — Residuals')

plt.suptitle('Residuals Distribution — CS (MPa)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('saved_models/comparison_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 4. Metrics summary table ─────────────────────────────────────────────────
print('\n===== Final Metrics Summary =====')
print(metrics_df.to_string())
print(f'\nBest R²:   {metrics_df["R²"].idxmax()} ({metrics_df["R²"].max():.4f})')
print(f'Best MAE:  {metrics_df["MAE"].idxmin()} ({metrics_df["MAE"].min():.4f})')
print(f'Best RMSE: {metrics_df["RMSE"].idxmin()} ({metrics_df["RMSE"].min():.4f})')